In [1]:
import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [2]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '2'
import numexpr as ne

In [3]:
!pip install -r requirements.txt

In [4]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
import openai
openai.api_key = userdata.get('OPENAI_API_KEY')

In [5]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
andrew_gina_docs = SimpleDirectoryReader(input_files=["./assets/AndrewHuberman/sleep/115_Dr_Gina_Poe_Use_Sleep_to_Enhance_Learning_Memory_&_Emotional_State_Huberman_Lab_Podcast.txt"], filename_as_id=True).load_data()
vector_index = VectorStoreIndex.from_documents(andrew_gina_docs)

In [6]:
vector_engine = vector_index.as_chat_engine()

In [8]:
from llama_index.core import ListIndex, Settings
from llama_index.core.node_parser import SimpleFileNodeParser

Settings.chunk_size = 1024
node_parser = SimpleFileNodeParser()
nodes = node_parser.get_nodes_from_documents(andrew_gina_docs)
list_index = ListIndex(nodes)

In [9]:
list_query_engine = list_index.as_query_engine(
    response_mode = "tree_summarize", use_async = True
)

In [10]:
from llama_index.core.tools.query_engine import QueryEngineTool

list_tool = QueryEngineTool.from_defaults(
    query_engine = list_query_engine,
    description="Useful for summarisation of the podcast about sleep and memory with dr. Gina Poe"
)
vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_engine,
    description="Useful fro retrieving of specific content about sleep and memory in the podcast topic"
)

In [15]:
from llama_index.core.selectors.pydantic_selectors import PydanticSingleSelector
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine

query_engine = RouterQueryEngine(
    selector=PydanticSingleSelector.from_defaults(),
    query_engine_tools=[
        list_tool,
        vector_tool
    ]
)

In [12]:
from llama_index.core.selectors.llm_selectors import LLMSingleSelector

query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        list_tool,
        vector_tool
    ]
)

In [19]:
openai.log = "debug"

In [20]:
from llama_index.core.response.pprint_utils import pprint_response
response = query_engine.query("Give me a quick summary of the Andrew Huberman podcast with dr. Gina Poe.")
pprint_response(response, show_source=True)

Final Response: The Andrew Huberman podcast with Dr. Gina Poe covers
various aspects of the relationship between sleep and its impact on
mental health, physical health, and performance. They discuss the
architecture of sleep, the role of the locus ceruleus in sleep
disturbances, the connection between sleep and trauma, the effects of
opiates on sleep, and the importance of sleep in addiction recovery.
Dr. Poe emphasizes the significance of maintaining a regular bedtime,
the impact of sleep disturbances on learning and memory, and the
potential for sleep interventions to assist in addiction recovery.
______________________________________________________________________
Source Node 1/1
Node ID: 634c2494-f292-497d-a640-748f299701c8
Similarity: None
Text: Welcome to the Huberman Lab podcast, where we discuss science
and science-based tools for everyday life, I'm Andrew Huberman and I'm
a professor of neurobiology and Ophthalmology at Stanford school of
medicine. Today my guest is Dr Gina 

In [18]:
import nest_asyncio
nest_asyncio.apply()